In [1]:
import numpy as np
import pandas as pd

print(f"NumPy {np.__version__}")
print(f"Pandas {pd.__version__}")

NumPy 2.4.2
Pandas 3.0.1


# Arrays, Lists, NumPy, and Pandas

This notebook has two goals:

1. Understand how Python lists differ from NumPy arrays, and why NumPy is useful for numerical work.
2. Practice a repeatable Pandas workflow: load, inspect, clean, filter, and transform tabular data.

The code uses a small in-memory dataset so it runs without downloading a file.

## 1. Python lists vs. NumPy arrays

A **list** is a general-purpose Python container. It can mix types and is excellent for collecting objects.

A **NumPy array** is designed for numerical data. It normally contains one data type, stores values compactly, and supports fast element-wise operations.

Key difference: `list_a * 2` repeats a list, while `array_a * 2` multiplies every number by 2.

In [2]:
python_scores = [60, 70, 80, 90]
numpy_scores = np.array(python_scores)

print("List * 2:  ", python_scores * 2)
print("Array * 2: ", numpy_scores * 2)
print("Array + 5: ", numpy_scores + 5)
print("Mean:      ", numpy_scores.mean())
print("Shape:     ", numpy_scores.shape)
print("Dtype:     ", numpy_scores.dtype)

List * 2:   [60, 70, 80, 90, 60, 70, 80, 90]
Array * 2:  [120 140 160 180]
Array + 5:  [65 75 85 95]
Mean:       75.0
Shape:      (4,)
Dtype:      int64


## 2. Pandas for tabular data

Pandas provides two core objects:

- A **Series** is one labeled column.
- A **DataFrame** is a table made from multiple aligned Series.

A typical workflow is:

`load -> inspect -> clean -> filter -> transform -> summarize`

For a real CSV, loading starts with `pd.read_csv("path/to/file.csv")`. Here we use `pd.DataFrame` to make the example self-contained.

In [3]:
raw_sales = pd.DataFrame({
    "product": ["Notebook", "Pen", "Notebook", "Mug", "Pen", "Mug"],
    "units": [12, 25, None, 8, 30, 10],
    "price": [4.50, 1.20, 4.50, 7.00, 1.20, None],
    "status": ["complete", "complete", "pending", "complete", "returned", "complete"],
    "customer": ["Asha", "Ben", "Cara", "Dev", "Asha", "Ben"],
})

print(raw_sales.head())
print("\nShape:", raw_sales.shape)
print("\nColumn types:\n", raw_sales.dtypes)

    product  units  price    status customer
0  Notebook   12.0    4.5  complete     Asha
1       Pen   25.0    1.2  complete      Ben
2  Notebook    NaN    4.5   pending     Cara
3       Mug    8.0    7.0  complete      Dev
4       Pen   30.0    1.2  returned     Asha

Shape: (6, 5)

Column types:
 product         str
units       float64
price       float64
status          str
customer        str
dtype: object


### Clean the table

Cleaning choices should reflect the meaning of the data. We will:

- remove duplicate rows if they exist;
- fill missing `units` with 0 because no recorded units should not become an invented sale;
- fill missing `price` with the median price;
- normalize text columns with `str.strip()` and `str.lower()`;
- convert numeric columns explicitly so calculations are predictable.

In [4]:
sales = raw_sales.copy().drop_duplicates()

sales["product"] = sales["product"].str.strip().str.lower()
sales["status"] = sales["status"].str.strip().str.lower()
sales["customer"] = sales["customer"].str.strip()
sales["units"] = pd.to_numeric(sales["units"], errors="coerce").fillna(0).astype(int)
sales["price"] = pd.to_numeric(sales["price"], errors="coerce")
sales["price"] = sales["price"].fillna(sales["price"].median())

print(sales)
print("\nMissing values after cleaning:\n", sales.isna().sum())

    product  units  price    status customer
0  notebook     12    4.5  complete     Asha
1       pen     25    1.2  complete      Ben
2  notebook      0    4.5   pending     Cara
3       mug      8    7.0  complete      Dev
4       pen     30    1.2  returned     Asha
5       mug     10    4.5  complete      Ben

Missing values after cleaning:
 product     0
units       0
price       0
status      0
customer    0
dtype: int64


### Filter and transform

Pandas filters use boolean conditions. Combine conditions with `&` and wrap each condition in parentheses. A transformation creates a new column from existing columns.

In [5]:
completed_sales = sales[sales["status"] == "complete"].copy()
large_completed_sales = completed_sales[completed_sales["units"] >= 10]

completed_sales["revenue"] = completed_sales["units"] * completed_sales["price"]

print("Completed sales with at least 10 units:\n", large_completed_sales)
print("\nRevenue by product:\n")
print(
    completed_sales.groupby("product", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)
print("\nOverall completed revenue:", completed_sales["revenue"].sum())

Completed sales with at least 10 units:
     product  units  price    status customer
0  notebook     12    4.5  complete     Asha
1       pen     25    1.2  complete      Ben
5       mug     10    4.5  complete      Ben

Revenue by product:

    product  revenue
0       mug    101.0
1  notebook     54.0
2       pen     30.0

Overall completed revenue: 185.0


## Practice

Using `sales`, complete these tasks:

1. Find all rows where the customer is `Ben`.
2. Add a `unit_price_after_discount` column using a 10% discount.
3. Calculate total completed units for each customer with `groupby`.
4. Explain why the pending row has 0 units after cleaning. Would a different business rule be better?

Useful tools: `.loc[...]`, `.assign(...)`, `.groupby(...)`, `.agg(...)`, `.sort_values(...)`.

In [6]:
ben_sales = sales.loc[sales["customer"] == "Ben"]
completed_customer_units = (
    completed_sales.groupby("customer", as_index=False)["units"]
    .sum()
    .sort_values("units", ascending=False)
)

discounted_sales = sales.assign(
    unit_price_after_discount=sales["price"] * 0.90
)

print("Ben's sales:\n", ben_sales)
print("\nCompleted units by customer:\n", completed_customer_units)
print("\nDiscounted prices:\n", discounted_sales[["product", "price", "unit_price_after_discount"]])

Ben's sales:
   product  units  price    status customer
1     pen     25    1.2  complete      Ben
5     mug     10    4.5  complete      Ben

Completed units by customer:
   customer  units
1      Ben     35
0     Asha     12
2      Dev      8

Discounted prices:
     product  price  unit_price_after_discount
0  notebook    4.5                       4.05
1       pen    1.2                       1.08
2  notebook    4.5                       4.05
3       mug    7.0                       6.30
4       pen    1.2                       1.08
5       mug    4.5                       4.05
